# Lab 08 solution, notebook 2: one-way transfers, and a manager with a session in a file

The worked answer to the challenge of `08b_google_adk_at_the_desk.ipynb`. The setup repeats what the notebook built
(`llm`, `reply_text`, `ask`, the three lookups, `show_run`, `model_calls` and the two specialists), so this file
runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from google.adk.agents import Agent
from google.adk.integrations.openai import OpenAIResponsesLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

llm = OpenAIResponsesLlm(model=MODEL)


def reply_text(event):
    """The text of an event: its text parts joined, leaving out the model's thoughts."""
    return "".join(part.text for part in event.content.parts if part.text and not part.thought)


async def ask(runner, session, text):
    """Send one user message in a session and return the events of the run."""
    message = types.Content(role="user", parts=[types.Part(text=text)])
    events = []
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id, new_message=message):
        events.append(event)
    return events

In [3]:
import sqlite3

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

In [4]:
def show_run(events):
    """Print what each event holds, with its number and its author in front."""
    for number, event in enumerate(events, start=1):
        for call in event.get_function_calls():
            print(f"{number:>2} {event.author:12} asks for   {call.name} {call.args}")
        for response in event.get_function_responses():
            print(f"{number:>2} {event.author:12} gets back  {response.name} {response.response}")
        if reply_text(event):
            print(f"{number:>2} {event.author:12} says       {reply_text(event)}")


def model_calls(events):
    """The input tokens of each model call in a list of events."""
    return [event.usage_metadata.prompt_token_count for event in events if event.usage_metadata]

In [5]:
invoices = Agent(
    name="invoices",
    model=llm,
    description="Answers questions about a customer's invoices: how many, and their total.",
    instruction=("You answer questions about a customer's invoices. Look up every fact with your tools; "
                 "never answer from memory. Answer in one or two sentences."),
    tools=[find_customer, get_invoices],
)

support_reps = Agent(
    name="support_reps",
    model=llm,
    description="Answers who a customer's support rep is.",
    instruction=("You answer who looks after a customer. Look up every fact with your tools; "
                 "never answer from memory. Answer in one or two sentences."),
    tools=[find_customer, get_support_rep],
)

## Task 1: make the transfers one way

`clone(update=...)` copies an agent and changes the fields we name. Both switches are set on both copies, so a
specialist that gets the conversation cannot pass it to the other specialist or back to `triage`.

In [6]:
ONE_WAY = {"disallow_transfer_to_parent": True, "disallow_transfer_to_peers": True}

one_way_triage = Agent(
    name="triage",
    model=llm,
    instruction=("You route each request to the right specialist. Transfer to invoices for invoice counts and "
                 "totals, and to support_reps for who looks after a customer."),
    sub_agents=[invoices.clone(update=ONE_WAY), support_reps.clone(update=ONE_WAY)],
)

PUJA = "How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?"

one_way_runner = InMemoryRunner(agent=one_way_triage, app_name="chinook_one_way")
one_way_session = await one_way_runner.session_service.create_session(app_name="chinook_one_way", user_id="rep")

events = await ask(one_way_runner, one_way_session, PUJA)

show_run(events)
print("model calls:", len(model_calls(events)))

App "chinook_one_way" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


 1 triage       asks for   transfer_to_agent {'agent_name': 'invoices'}
 1 triage       asks for   transfer_to_agent {'agent_name': 'support_reps'}
 2 triage       gets back  transfer_to_agent {'result': None}
 2 triage       gets back  transfer_to_agent {'result': None}
 3 support_reps asks for   find_customer {'name': 'Puja Srivastava'}
 4 support_reps gets back  find_customer {'result': [{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}]}
 5 support_reps asks for   get_support_rep {'customer_id': 59}
 6 support_reps gets back  get_support_rep {'customer_id': 59, 'support_rep': 'Jane Peacock'}
 7 support_reps says       Puja Srivastava’s support rep is Jane Peacock. I can’t verify her invoice count or total with the available tools.
model calls: 4


The conversation went to `support_reps` and stayed there. (`triage` asked for two transfers again, and ADK took
the one named last.) `support_reps` looked up the rep, Jane Peacock, and with both switches set it could not pass
the rest on. Its reply says it cannot verify the invoice count or total. The run made four model calls and raised no error,
and the reply reads well. It still has no invoice facts.

One more rule of ADK, which this run does not show: with `disallow_transfer_to_parent` set, the next question
goes back to `triage`, not to a specialist that cannot pass the conversation on.

Compare this with Part C. Letting the specialists transfer cost more model calls and gave the whole answer.
Stopping them costs less and loses whatever the first specialist cannot look up. A team with one-way transfers
fits requests that one specialist can answer alone.

## Task 2: carry a conversation through the manager, in a file

The manager gets a `Runner` with a `SqliteSessionService`, so its session is stored in a file under `outputs/`.
The specialists here belong to no team yet, so `AgentTool` can take them as they are.

In [7]:
from google.adk.runners import Runner
from google.adk.sessions.sqlite_session_service import SqliteSessionService
from google.adk.tools import AgentTool

manager = Agent(
    name="manager",
    model=llm,
    instruction=("You help the support reps of a music shop. Ask the specialists for every fact; never answer from "
                 "memory. Answer in one or two sentences."),
    tools=[AgentTool(agent=invoices), AgentTool(agent=support_reps)],
)

SESSION_FILE = Path("outputs") / "manager_sessions.db"
SESSION_FILE.parent.mkdir(exist_ok=True)
SESSION_FILE.unlink(missing_ok=True)

service = SqliteSessionService(db_path=str(SESSION_FILE))
manager_runner = Runner(agent=manager, app_name="chinook_manager", session_service=service)
manager_session = await service.create_session(app_name="chinook_manager", user_id="rep", session_id="rep-puja")

events = await ask(manager_runner, manager_session, PUJA)
print(reply_text(events[-1]))
print()

events = await ask(manager_runner, manager_session, "And what about Helena Holý?")
show_run(events)

Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.



 1 manager      asks for   invoices {'request': 'How many invoices does Helena Holý have, and what is their total amount?'}
 1 manager      asks for   support_reps {'request': "Who is Helena Holý's support rep?"}
 2 manager      gets back  invoices {'result': 'Helena Holý has 7 invoices totaling $49.62.'}
 2 manager      gets back  support_reps {'result': 'Helena Holý’s support rep is Steve Johnson.'}
 3 manager      says       Helena Holý has 7 invoices totaling $49.62, and her support rep is Steve Johnson.


The first reply has Puja's three facts. For the follow-up, read the two `asks for` lines. The rep's message said
only *And what about Helena Holý?*, and the manager wrote a full request for each specialist, with her name in it.
A specialist sees only that request, never the session, so the manager has to put everything the specialist needs
into it. The answer has 7 invoices, \$49.62 and Steve Johnson.

The conversation is stored in `outputs/manager_sessions.db`, under the id `rep-puja`.